# Continuous-Time Quantum Walk on $P_4 \,\square\, P_4$ and Punctured Variants

This notebook visualizes the **continuous-time dynamics** of quantum walks on the line graph of $P_4 \square P_4$ and its punctured variants. Unlike the discrete average mixing matrix $\widehat{M}$, here we plot $M(t) = U(t) \circ U(-t)$ as a function of $t$ to see how the walk spreads, oscillates, and (in the limit) approaches the time-averaged behavior.

We produce three types of visualizations:
1. **Time-series plots** — diagonal entries $M(t)_{q,q}$ vs $t$, showing oscillations.
2. **Animated heatmaps** — $M(t)$ as a function of $t$, with time slider.
3. **Animated graph-edge weights** — the walker's probability distribution over edges of $\Gamma$ at time $t$, drawn on the grid.

All three illustrate **how the walk approaches its long-time average** — and where it doesn't (genuinely oscillatory components persist forever in finite-dimensional systems).

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

np.set_printoptions(precision=4, suppress=True, linewidth=160)
plt.rcParams['animation.html'] = 'jshtml'  # for inline animation in jupyter

## 1. Helpers: line graph, time evolution, and time-averaged comparison

In [ ]:
def line_graph_adjacency(G):
    LG = nx.line_graph(G)
    edge_list = [tuple(sorted(e)) for e in LG.nodes()]
    edge_index = {e: i for i, e in enumerate(edge_list)}
    m = len(edge_list)
    A = np.zeros((m, m))
    for u, v in LG.edges():
        i, j = edge_index[tuple(sorted(u))], edge_index[tuple(sorted(v))]
        A[i, j] = 1.0
        A[j, i] = 1.0
    return A, edge_list

def time_evolved_M(A, t_array):
    """
    For each t in t_array, returns M(t) = |U(t)|^2 entrywise (real non-negative).
    Uses spectral decomposition once and evaluates U(t) for each t cheaply.
    """
    eigvals, eigvecs = np.linalg.eigh(A)
    Vt = eigvecs.T  # shape (m, m), rows = eigenvectors
    Ms = []
    for t in t_array:
        # U(t)_{ij} = sum_r eigvec[i,r] * exp(i t lam_r) * eigvec[j,r]
        Dt = np.exp(1j * t * eigvals)
        U = (eigvecs * Dt) @ eigvecs.T
        Ms.append(np.abs(U)**2)
    return np.array(Ms)  # shape (len(t_array), m, m)

def average_mixing_matrix(A, atol=1e-9):
    """M_hat = sum_r E_r ∘ E_r where A = sum_r theta_r E_r."""
    eigvals, eigvecs = np.linalg.eigh(A)
    grouped = []
    used = np.zeros(len(eigvals), dtype=bool)
    for i, lam in enumerate(eigvals):
        if used[i]:
            continue
        group = [i]
        used[i] = True
        for j in range(i+1, len(eigvals)):
            if not used[j] and abs(eigvals[j]-lam) < atol:
                group.append(j)
                used[j] = True
        V = eigvecs[:, group]
        grouped.append((lam, V @ V.T))
    M_hat = np.zeros_like(A)
    for lam, E_r in grouped:
        M_hat += E_r * E_r
    return M_hat

## 2. Build $P_4 \square P_4$ and compute time evolution

In [ ]:
G = nx.grid_2d_graph(4, 4)
n = G.number_of_nodes()
m = G.number_of_edges()
A, edge_list = line_graph_adjacency(G)
node_list = list(G.nodes())
node_index = {v: i for i, v in enumerate(node_list)}

# Choose a starting edge — corner edge
start_edge = ((0, 0), (0, 1))
q = edge_list.index(tuple(sorted(start_edge)))
print(f'Starting edge: {start_edge} (index {q})')
print(f'Graph: n = {n}, m = {m}')

# Choose time grid
T_max = 30.0
n_frames = 300
t_array = np.linspace(0, T_max, n_frames)

# Compute M(t) at each time
Ms = time_evolved_M(A, t_array)
print(f'Computed M(t) at {n_frames} time points, t in [0, {T_max}]')
print(f'Each M(t) is {m}x{m}')

# Average mixing matrix (long-time limit, for comparison)
M_hat = average_mixing_matrix(A)

## 3. Time-series: how $M(t)_{q,q}$ oscillates and approaches $\widehat{M}_{q,q}$

For the starting edge $q$, we plot the diagonal entry $M(t)_{q,q}$ — the probability of being back at $q$ at time $t$. We compare against the time-averaged value $\widehat{M}_{q,q}$ (horizontal line).

In [ ]:
# Plot M(t)[q,p] for p = q, and a few other edges
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Top: return probability M(t)[q,q]
ax = axes[0]
ax.plot(t_array, Ms[:, q, q], lw=1.2, label=f'$M(t)_{{q,q}}$  (return to start)')
ax.axhline(M_hat[q, q], color='red', ls='--', lw=1.5,
           label=f'$\\widehat{{M}}_{{q,q}} = {M_hat[q,q]:.4f}$')
ax.axhline(1/m, color='green', ls=':', lw=1.2,
           label=f'$1/m = {1/m:.4f}$ (uniform)')
ax.set_ylabel('Return probability')
ax.set_title(f'Quantum walk on $\\ell(P_4  x  P_4)$ starting from edge {start_edge}')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

# Bottom: probability at a few other edges
ax = axes[1]
other_edges = [
    ((0, 1), (0, 2)),    # adjacent
    ((1, 1), (1, 2)),    # interior
    ((3, 3), (3, 2)),    # opposite corner area
]
for e in other_edges:
    p = edge_list.index(tuple(sorted(e)))
    ax.plot(t_array, Ms[:, p, q], lw=1, alpha=0.8, label=f'edge {e}')
    ax.axhline(M_hat[p, q], ls='--', lw=0.8, alpha=0.5)
ax.set_xlabel('time $t$')
ax.set_ylabel('Probability $M(t)_{p,q}$')
ax.set_title('Probability at other edges (dashed = time average)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Observations:**
- The return probability $M(t)_{q,q}$ oscillates rapidly and does **not** converge to $\widehat{M}_{q,q}$.
- This is because the walk is unitary: it has no decoherence, and $M(t)$ is almost-periodic (a sum of complex exponentials).
- The time average $\widehat{M}_{q,q}$ is a *Cesàro mean*, not a pointwise limit.
- Different edges have very different oscillation amplitudes: nearby edges show strong response; far edges respond weakly with delay.

## 4. Animated heatmap: $M(t)$ as $t$ varies

We animate the full mixing matrix $M(t)$ side-by-side with the time-averaged $\widehat{M}$ (which is static). This shows: at any single moment, the walk is concentrated on a few edges; only after averaging do we see uniform spreading.

In [ ]:
# Animation of M(t) heatmap
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

vmax = max(Ms.max(), M_hat.max()) * 0.5  # show with reasonable color scale

im_left = axes[0].imshow(Ms[0], cmap='viridis', vmin=0, vmax=vmax, aspect='equal')
title_left = axes[0].set_title(f'$M(t)$ at $t = 0.00$')
plt.colorbar(im_left, ax=axes[0], shrink=0.8)

im_right = axes[1].imshow(M_hat, cmap='viridis', vmin=0, vmax=vmax, aspect='equal')
axes[1].set_title(f'time-averaged $\\widehat{{M}}$')
plt.colorbar(im_right, ax=axes[1], shrink=0.8)

for ax in axes:
    ax.set_xlabel('edge index')
    ax.set_ylabel('edge index')

def update(frame):
    im_left.set_data(Ms[frame])
    title_left.set_text(f'$M(t)$ at $t = {t_array[frame]:.2f}$')
    return im_left, title_left

anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.tight_layout()
plt.close(fig)  # don't show static figure
HTML(anim.to_jshtml())

## 5. Animated walk on the grid

Now we visualize the walk **on the original graph $\Gamma$**, drawing edge probabilities as colors. This is the most intuitive picture: you can see the wave-like spreading of the quantum walk.

In [ ]:
# Setup positions and edges
pos = {(i, j): (j, -i) for i, j in G.nodes()}  # row index = y (negated), col = x
edges_in_G = list(G.edges())
edge_idx_in_LG = [edge_list.index(tuple(sorted(e))) for e in edges_in_G]
segments = [[pos[u], pos[v]] for (u, v) in edges_in_G]

fig, ax = plt.subplots(figsize=(7, 7))

# Use 95th percentile of post-initial weights for color scale —
# this gives good dynamic range during wave spreading
# (initial frame has all weight on one edge with value ~1, which would
#  saturate the colormap if used as vmax)
weights_after_t0 = Ms[1:, :, q].flatten()
vmax_grid = float(np.percentile(weights_after_t0, 95))
print(f'vmax_grid = {vmax_grid:.4f} (95th percentile of post-initial weights)')

from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
cmap = plt.cm.plasma
norm = Normalize(vmin=0, vmax=vmax_grid)

# Draw nodes (static)
nx.draw_networkx_nodes(G, pos=pos, node_color='lightgray', node_size=400, ax=ax)
nx.draw_networkx_labels(G, pos=pos, font_size=8, ax=ax)

# Highlight starting edge with a red overlay
nx.draw_networkx_edges(G, pos=pos, edgelist=[start_edge],
                       edge_color='red', width=10, alpha=0.4, ax=ax)

# Animated edges as a manual LineCollection (this is the key for proper animation)
lc = LineCollection(segments, linewidths=4, cmap=cmap, norm=norm)
lc.set_array(np.array([Ms[0, idx, q] for idx in edge_idx_in_LG]))
ax.add_collection(lc)

ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-3.5, 0.5)
ax.set_aspect('equal')
title = ax.set_title(f'Quantum walk on $P_4 \\times P_4$\nstart: {start_edge},   $t = 0.00$')
ax.set_axis_off()

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm._A = []
plt.colorbar(sm, ax=ax, label=r'$M(t)_{p,q}$', shrink=0.7, extend='max')

def update_grid(frame):
    weights = np.array([Ms[frame, idx, q] for idx in edge_idx_in_LG])
    lc.set_array(weights)
    title.set_text(f'Quantum walk on $P_4 \\times P_4$\nstart: {start_edge},   $t = {t_array[frame]:.2f}$')
    return [lc, title]

anim_grid = FuncAnimation(fig, update_grid, frames=n_frames, interval=50, blit=False)
plt.tight_layout()
plt.close(fig)
HTML(anim_grid.to_jshtml())

## 6. Same simulation on a punctured grid

Now we do the same on $P_4 \square P_4$ minus the interior vertex $(1,1)$. Watch how the wave **routes around the hole** — quantum walks have interesting interference patterns near defects.

In [ ]:
G_punc = G.copy()
G_punc.remove_node((1, 1))
n_punc = G_punc.number_of_nodes()
m_punc = G_punc.number_of_edges()
A_punc, edge_list_punc = line_graph_adjacency(G_punc)
M_hat_punc = average_mixing_matrix(A_punc)
print(f'Punctured grid: n = {n_punc}, m = {m_punc}')
print(f'Connected: {nx.is_connected(G_punc)}')

# Same starting edge if it still exists
if tuple(sorted(start_edge)) in edge_list_punc:
    q_punc = edge_list_punc.index(tuple(sorted(start_edge)))
    print(f'Starting edge: {start_edge} (index {q_punc} in punctured graph)')
else:
    raise ValueError('Starting edge no longer exists in punctured graph')

Ms_punc = time_evolved_M(A_punc, t_array)

In [ ]:
# Time-series comparison: full grid vs punctured
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t_array, Ms[:, q, q], lw=1.2, alpha=0.85, label=f'full grid')
ax.plot(t_array, Ms_punc[:, q_punc, q_punc], lw=1.2, alpha=0.85, label=f'minus (1,1)')
ax.axhline(M_hat[q, q], color='C0', ls='--', alpha=0.5,
           label=f'full: $\\widehat{{M}}_{{q,q}} = {M_hat[q,q]:.4f}$')
ax.axhline(M_hat_punc[q_punc, q_punc], color='C1', ls='--', alpha=0.5,
           label=f'punc: $\\widehat{{M}}_{{q,q}} = {M_hat_punc[q_punc,q_punc]:.4f}$')
ax.set_xlabel('time $t$')
ax.set_ylabel('return probability $M(t)_{q,q}$')
ax.set_title(f'Return probability at edge {start_edge}: full vs punctured grid')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Animated walk on punctured grid (using LineCollection for proper animation)
pos_punc = {(i, j): (j, -i) for i, j in G_punc.nodes()}
edges_in_Gpunc = list(G_punc.edges())
edge_idx_punc = [edge_list_punc.index(tuple(sorted(e))) for e in edges_in_Gpunc]
segments_punc = [[pos_punc[u], pos_punc[v]] for (u, v) in edges_in_Gpunc]

fig, ax = plt.subplots(figsize=(7, 7))

weights_after_t0_punc = Ms_punc[1:, :, q_punc].flatten()
vmax_grid_punc = float(np.percentile(weights_after_t0_punc, 95))
norm_punc = Normalize(vmin=0, vmax=vmax_grid_punc)

# Draw nodes
nx.draw_networkx_nodes(G_punc, pos=pos_punc, node_color='lightgray',
                       node_size=400, ax=ax)
# Mark removed vertex with red X
ax.plot(pos[(1,1)][0], pos[(1,1)][1], 'rx', markersize=20, mew=3)
nx.draw_networkx_labels(G_punc, pos=pos_punc, font_size=8, ax=ax)

# Highlight starting edge
nx.draw_networkx_edges(G_punc, pos=pos_punc, edgelist=[start_edge],
                       edge_color='red', width=10, alpha=0.4, ax=ax)

# Animated LineCollection
lc_punc = LineCollection(segments_punc, linewidths=4, cmap=cmap, norm=norm_punc)
lc_punc.set_array(np.array([Ms_punc[0, idx, q_punc] for idx in edge_idx_punc]))
ax.add_collection(lc_punc)

ax.set_xlim(-0.5, 3.5); ax.set_ylim(-3.5, 0.5); ax.set_aspect('equal')
title_punc = ax.set_title(f'Walk on punctured $P_4 \\times P_4$ (minus (1,1))\nstart: {start_edge},   $t = 0.00$')
ax.set_axis_off()

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_punc); sm._A = []
plt.colorbar(sm, ax=ax, label=r'$M(t)_{p,q}$', shrink=0.7, extend='max')

def update_punc(frame):
    weights = np.array([Ms_punc[frame, idx, q_punc] for idx in edge_idx_punc])
    lc_punc.set_array(weights)
    title_punc.set_text(f'Walk on punctured $P_4 \\times P_4$ (minus (1,1))\nstart: {start_edge},   $t = {t_array[frame]:.2f}$')
    return [lc_punc, title_punc]

anim_punc = FuncAnimation(fig, update_punc, frames=n_frames, interval=50, blit=False)
plt.tight_layout()
plt.close(fig)
HTML(anim_punc.to_jshtml())

## 7. Side-by-side animation: full vs punctured grid

The most direct comparison: same starting edge, same time grid, watch both walks evolve. Look for:
- **Diffraction patterns** near the missing vertex.
- **Localization** — energy gets trapped on certain edges.
- **Different oscillation frequencies** due to different spectra.

In [ ]:
# Side-by-side animation: full grid vs punctured (LineCollection version)
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))

# Use a shared color scale based on both walks
vmax_compare = float(max(
    np.percentile(Ms[1:, :, q].flatten(), 95),
    np.percentile(Ms_punc[1:, :, q_punc].flatten(), 95)
))
norm_compare = Normalize(vmin=0, vmax=vmax_compare)

# === Left panel: full grid ===
ax = axes[0]
nx.draw_networkx_nodes(G, pos=pos, node_color='lightgray', node_size=350, ax=ax)
nx.draw_networkx_edges(G, pos=pos, edgelist=[start_edge],
                       edge_color='red', width=8, alpha=0.4, ax=ax)
lc_full = LineCollection(segments, linewidths=4, cmap=cmap, norm=norm_compare)
lc_full.set_array(np.array([Ms[0, idx, q] for idx in edge_idx_in_LG]))
ax.add_collection(lc_full)
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-3.5, 0.5); ax.set_aspect('equal')
title_full = ax.set_title(f'Full $P_4 \\times P_4$  ($t = 0.00$)')
ax.set_axis_off()

# === Right panel: punctured ===
ax = axes[1]
nx.draw_networkx_nodes(G_punc, pos=pos_punc, node_color='lightgray',
                       node_size=350, ax=ax)
ax.plot(pos[(1,1)][0], pos[(1,1)][1], 'rx', markersize=18, mew=3)
nx.draw_networkx_edges(G_punc, pos=pos_punc, edgelist=[start_edge],
                       edge_color='red', width=8, alpha=0.4, ax=ax)
lc_punc2 = LineCollection(segments_punc, linewidths=4, cmap=cmap, norm=norm_compare)
lc_punc2.set_array(np.array([Ms_punc[0, idx, q_punc] for idx in edge_idx_punc]))
ax.add_collection(lc_punc2)
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-3.5, 0.5); ax.set_aspect('equal')
title_punc2 = ax.set_title(f'Minus (1,1)  ($t = 0.00$)')
ax.set_axis_off()

# Shared colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_compare); sm._A = []
fig.colorbar(sm, ax=axes, label=r'$M(t)_{p,q}$', shrink=0.7, pad=0.02, extend='max')
fig.suptitle(f'Quantum walk starting from edge {start_edge}', fontsize=12, y=1.02)

def update_compare(frame):
    lc_full.set_array(np.array([Ms[frame, idx, q] for idx in edge_idx_in_LG]))
    title_full.set_text(f'Full $P_4 \\times P_4$  ($t = {t_array[frame]:.2f}$)')
    lc_punc2.set_array(np.array([Ms_punc[frame, idx, q_punc] for idx in edge_idx_punc]))
    title_punc2.set_text(f'Minus (1,1)  ($t = {t_array[frame]:.2f}$)')
    return [lc_full, lc_punc2, title_full, title_punc2]

anim_compare = FuncAnimation(fig, update_compare, frames=n_frames, interval=50, blit=False)
plt.close(fig)
HTML(anim_compare.to_jshtml())

## 8. Long-time behavior: running average $\frac{1}{T}\int_0^T M(t)\,dt$

This makes the convergence to $\widehat{M}$ visually explicit. Even though $M(t)$ never settles, its **running average** does converge.

In [ ]:
# Running average of M(t)[q,q]
running_avg = np.cumsum(Ms[:, q, q]) / np.arange(1, n_frames + 1)
running_avg_punc = np.cumsum(Ms_punc[:, q_punc, q_punc]) / np.arange(1, n_frames + 1)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t_array, Ms[:, q, q], lw=0.7, alpha=0.4, color='C0', label=f'$M(t)_{{q,q}}$ (full)')
ax.plot(t_array, running_avg, lw=2.5, color='C0', label=f'running avg (full)')
ax.axhline(M_hat[q, q], color='C0', ls='--', alpha=0.6,
           label=f'full: $\\widehat{{M}}_{{q,q}} = {M_hat[q,q]:.4f}$')

ax.plot(t_array, Ms_punc[:, q_punc, q_punc], lw=0.7, alpha=0.4, color='C1', label=f'$M(t)_{{q,q}}$ (punc)')
ax.plot(t_array, running_avg_punc, lw=2.5, color='C1', label=f'running avg (punc)')
ax.axhline(M_hat_punc[q_punc, q_punc], color='C1', ls='--', alpha=0.6,
           label=f'punc: $\\widehat{{M}}_{{q,q}} = {M_hat_punc[q_punc,q_punc]:.4f}$')

ax.set_xlabel('time $t$')
ax.set_ylabel('probability')
ax.set_title('Instantaneous return $M(t)_{q,q}$ vs running time-average\n(approaches $\\widehat{M}_{q,q}$ but instantaneous oscillates forever)')
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Save key animations as GIF

Save GIFs that you can embed in slides directly. (Skip this cell if you don't need files saved to disk.)

In [ ]:
save_gifs = False  # set True to save GIFs

if save_gifs:
    print('Saving full grid animation...')
    anim.save('walk_full_grid.gif', writer=PillowWriter(fps=20))
    print('Saving punctured grid animation...')
    anim_punc.save('walk_punctured_grid.gif', writer=PillowWriter(fps=20))
    print('Saving comparison animation...')
    anim_compare.save('walk_comparison.gif', writer=PillowWriter(fps=20))
    print('Done. Files saved in current directory.')
else:
    print('save_gifs=False: not saving. Set to True to save GIFs.')

## 10. Summary: what continuous time tells us that $\widehat{M}$ doesn't

**The instantaneous mixing matrix $M(t)$ shows things the average $\widehat{M}$ hides:**

1. **Coherent oscillations.** $M(t)_{q,q}$ never settles — it's a superposition of sinusoids with frequencies determined by spectral gaps of $A(\ell\Gamma)$. The walk is reversible.

2. **Wave-like spreading.** On the grid, you can see the walker's amplitude propagate outward from the source edge in a wave pattern.

3. **Defect-induced scattering.** With a missing vertex, the wave **diffracts around the hole** — different from classical random walks, which simply have lower diffusion in punctured graphs. Quantum interference matters here.

4. **Localization signatures.** Some edges retain probability for surprisingly long times — these are signatures of states with strong overlap with localized eigenvectors.

**Connection to the Schur state framework:**
- The Schur state $S^e(t)$ for a pure edge state encodes all of this dynamical information.
- $A(e) = \overline{S^e}\circ S^e$ is precisely $M(t)$ restricted to one column (the column corresponding to $e$), reshaped onto the graph $\Gamma$.
- Time-averaging gives $\widehat{A(e)}_{v,w} = \widehat{M}_{e_{vw}, e}$ — the column of $\widehat{M}$ corresponding to $e$, displayed as edge weights on $\Gamma$.
- The instantaneous Schur state captures the **full** wave-function dynamics; averaging captures only the long-time return distribution.

**Why $\widehat{M}$ matters despite being a Cesàro-mean object:**
- For graph invariants and combinatorial questions (like spanning tree counting), only the time-averaged behavior is relevant — the oscillations average out.
- The main theorem $tn(\Gamma, 1/m) = m^{-(n-1)}\, tn(\Gamma)$ uses $\widehat{A(e)} = \frac{1}{m}A(\Gamma)$, which is a statement about $\widehat{M}$, not $M(t)$ directly.